# 🧠 SNN vs Transformer: Performance Benchmark
### Time · Accuracy · Energy Tradeoffs for Neuromorphic NLP
---
**Task**: AG News text classification (4 classes)
**Models**: Dense Transformer vs Recurrent SNN vs Pruned SNN
**Metrics**: Training time, inference latency, accuracy, energy (once)

In [ ]:
!pip install -q snntorch datasets transformers matplotlib seaborn numpy

In [ ]:
!pip install --upgrade datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.18.0
    Uninstalling huggingface_hub-1.18.0:
      Successfully uninstalled huggingface_hub-1.18.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoTokenizer
import snntorch as snn
from snntorch import surrogate
import torch.nn.utils.prune as prune
import time
import json
import copy
import numpy as np

# ─── Configuration ───
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {DEVICE}")

BATCH_SIZE = 64
MAX_SEQ_LEN = 32
EMBED_DIM = 64
HIDDEN_DIM = 128
NUM_CLASSES = 4
CLASS_NAMES = ['World', 'Sports', 'Business', 'Sci/Tech']
EPOCHS_MAIN = 5
EPOCHS_FINETUNE = 3

# Energy estimates (45nm CMOS) — used once for comparison
E_MAC_PJ = 4.6   # Multiply-Accumulate (pJ)
E_AC_PJ  = 0.9   # Accumulate only   (pJ)

🖥️  Device: cuda


## 📦 1. Dataset: AG News (4-class topic classification)

In [ ]:
print("Loading AG News dataset...")
dataset = load_dataset("fancyzhx/ag_news")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
VOCAB_SIZE = tokenizer.vocab_size

def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length",
                     truncation=True, max_length=MAX_SEQ_LEN)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized.set_format(type="torch", columns=["input_ids", "label"])

train_dataset = torch.utils.data.Subset(tokenized["train"], range(10000))
test_dataset  = torch.utils.data.Subset(tokenized["test"],  range(2000))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print(f"✅ Train: {len(train_dataset):,} samples | Test: {len(test_dataset):,} samples")
print(f"   Vocab: {VOCAB_SIZE:,} | Seq length: {MAX_SEQ_LEN}")

Loading AG News dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

✅ Train: 10,000 samples | Test: 2,000 samples
   Vocab: 30,522 | Seq length: 32


## 🏗️ 2. Model Architectures

In [ ]:
class MiniTransformer(nn.Module):
    # Dense self-attention — every token attends to every other (O(N²))
    # All operations are Multiply-Accumulate (MAC)
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        self.pos_emb = nn.Embedding(MAX_SEQ_LEN, EMBED_DIM)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=EMBED_DIM, nhead=4,
            dim_feedforward=HIDDEN_DIM, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc = nn.Linear(EMBED_DIM, NUM_CLASSES)

    def forward(self, x):
        B, S = x.shape
        pos = torch.arange(0, S, device=x.device).expand(B, S)
        out = self.emb(x) + self.pos_emb(pos)
        out = self.transformer(out)
        out = out.mean(dim=1)          # Global average pooling
        logits = self.fc(out)
        # Approximate MAC count
        macs_per_token = 4 * (EMBED_DIM ** 2) + 2 * (EMBED_DIM * HIDDEN_DIM)
        total_ops = B * S * macs_per_token
        return logits, total_ops

print(f"✅ MiniTransformer defined")

✅ MiniTransformer defined


In [ ]:
class RecurrentSNN(nn.Module):
    # Spiking Neural Network — processes tokens sequentially
    # Neurons fire binary spikes → weight × 1 = just Addition (AC)
    # If neuron doesn't fire → skip entirely (extreme sparsity)
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        spike_grad = surrogate.fast_sigmoid(slope=25)
        self.fc1 = nn.Linear(EMBED_DIM, HIDDEN_DIM)
        self.lif1 = snn.Leaky(beta=0.9, spike_grad=spike_grad, init_hidden=True)
        self.fc2 = nn.Linear(HIDDEN_DIM, NUM_CLASSES)
        self.lif2 = snn.Leaky(beta=0.9, spike_grad=spike_grad, init_hidden=True)

    def forward(self, x):
        B, S = x.shape
        emb = self.emb(x)
        self.lif1.init_leaky()
        self.lif2.init_leaky()
        spk2_sum = torch.zeros(B, NUM_CLASSES, device=x.device)
        total_ops = 0
        for t in range(S):
            cur = self.fc1(emb[:, t, :])
            spk1, _ = self.lif1(cur)
            cur_out = self.fc2(spk1)
            spk2, _ = self.lif2(cur_out)
            spk2_sum += spk2
            # AC ops: each spike triggers additions across fan-out
            total_ops += spk1.sum().item() * HIDDEN_DIM
            total_ops += spk2.sum().item() * NUM_CLASSES
        return spk2_sum, total_ops

print(f"✅ RecurrentSNN defined")

✅ RecurrentSNN defined


## ⏱️ 3. Training Benchmark
Tracks per-epoch: **loss**, **accuracy**, **wall-clock time**, and **energy**.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def count_nonzero_weights(model):
    total, alive = 0, 0
    for name, param in model.named_parameters():
        if 'weight' in name:
            total += param.numel()
            alive += (param != 0).sum().item()
    return total, alive

def train_and_profile(model, name, is_snn=False, epochs=5, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.to(DEVICE)

    n_params = count_parameters(model)
    history = {
        'name': name,
        'loss': [],
        'train_acc': [],
        'time_per_epoch_s': [],
        'energy_mJ': [],
        'total_params': n_params,
    }

    print(f"\n{'='*55}")
    print(f"  Training: {name}  ({n_params:,} parameters)")
    print(f"{'='*55}")

    total_start = time.time()

    for epoch in range(epochs):
        model.train()
        t0 = time.time()
        running_loss, correct, total_samples, total_energy_pJ = 0, 0, 0, 0

        for batch in train_loader:
            inputs = batch['input_ids'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            optimizer.zero_grad()

            logits, ops = model(inputs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            correct += (logits.argmax(1) == labels).sum().item()
            total_samples += labels.size(0)
            total_energy_pJ += ops * (E_AC_PJ if is_snn else E_MAC_PJ)

        epoch_time = time.time() - t0
        epoch_acc  = correct / total_samples
        epoch_loss = running_loss / len(train_loader)
        epoch_energy = total_energy_pJ / 1e9   # pJ → mJ

        history['loss'].append(round(epoch_loss, 4))
        history['train_acc'].append(round(epoch_acc, 4))
        history['time_per_epoch_s'].append(round(epoch_time, 2))
        history['energy_mJ'].append(round(epoch_energy, 2))

        print(f"  Epoch {epoch+1}/{epochs}  │  "
              f"Acc {epoch_acc:.4f}  │  Loss {epoch_loss:.4f}  │  "
              f"Time {epoch_time:.2f}s  │  Energy {epoch_energy:.2f} mJ")

    history['total_train_time_s'] = round(time.time() - total_start, 2)
    print(f"  ──────────────────────────────────────────────")
    print(f"  Total training time: {history['total_train_time_s']:.2f}s")

    return history

In [ ]:
transformer_model = MiniTransformer()
history_tf = train_and_profile(transformer_model, "Transformer",
                               is_snn=False, epochs=EPOCHS_MAIN)


  Training: Transformer  (1,989,188 parameters)
  Epoch 1/5  │  Acc 0.4792  │  Loss 1.1766  │  Time 1.84s  │  Energy 48.23 mJ
  Epoch 2/5  │  Acc 0.7145  │  Loss 0.7379  │  Time 0.88s  │  Energy 48.23 mJ
  Epoch 3/5  │  Acc 0.8060  │  Loss 0.5259  │  Time 0.88s  │  Energy 48.23 mJ
  Epoch 4/5  │  Acc 0.8575  │  Loss 0.3959  │  Time 0.88s  │  Energy 48.23 mJ
  Epoch 5/5  │  Acc 0.8986  │  Loss 0.2954  │  Time 0.89s  │  Energy 48.23 mJ
  ──────────────────────────────────────────────
  Total training time: 5.36s


In [ ]:
class RecurrentSNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        spike_grad = surrogate.fast_sigmoid(slope=25)
        self.fc1 = nn.Linear(EMBED_DIM, HIDDEN_DIM)
        self.lif1 = snn.Leaky(beta=0.9, spike_grad=spike_grad, init_hidden=True)
        self.fc2 = nn.Linear(HIDDEN_DIM, NUM_CLASSES)
        self.lif2 = snn.Leaky(beta=0.9, spike_grad=spike_grad, init_hidden=True)

    def forward(self, x):
        B, S = x.shape
        emb = self.emb(x)
        self.lif1.init_leaky()
        self.lif2.init_leaky()
        spk2_sum = torch.zeros(B, NUM_CLASSES, device=x.device)
        total_ops = 0
        for t in range(S):
            cur = self.fc1(emb[:, t, :])
            spk1 = self.lif1(cur) # Fixed unpacking
            cur_out = self.fc2(spk1)
            spk2 = self.lif2(cur_out) # Fixed unpacking
            spk2_sum += spk2
            total_ops += spk1.sum().item() * HIDDEN_DIM
            total_ops += spk2.sum().item() * NUM_CLASSES
        return spk2_sum, total_ops

snn_model = RecurrentSNN()
history_snn = train_and_profile(snn_model, "SNN",
                                is_snn=True, epochs=EPOCHS_MAIN)


  Training: SNN  (1,962,244 parameters)
  Epoch 1/5  │  Acc 0.3475  │  Loss 1.3111  │  Time 8.90s  │  Energy 0.31 mJ
  Epoch 2/5  │  Acc 0.5608  │  Loss 1.0389  │  Time 8.81s  │  Energy 0.31 mJ
  Epoch 3/5  │  Acc 0.6695  │  Loss 0.8308  │  Time 8.83s  │  Energy 0.32 mJ
  Epoch 4/5  │  Acc 0.7373  │  Loss 0.6895  │  Time 9.14s  │  Energy 0.32 mJ
  Epoch 5/5  │  Acc 0.7866  │  Loss 0.5739  │  Time 8.79s  │  Energy 0.32 mJ
  ──────────────────────────────────────────────
  Total training time: 44.47s


## ✂️ 4. Synaptic Pruning (Bio-Inspired Compression)
During brain development, ~50% of synapses are pruned to optimize efficiency.
We replicate this by removing the 50% lowest-magnitude weights.

In [ ]:
import torch
from torch.sparse import to_sparse_semi_structured

def apply_2_4_sparsity(linear_layer, layer_name="Layer"):
    """
    Enforces strict 2:4 sparsity and compresses the tensor
    to engage A100 Sparse Tensor Cores via cuSPARSELt.
    Includes hardware alignment constraints.
    """
    out_features, in_features = linear_layer.weight.shape

    # 1. HARDWARE GUARD: Ampere Tensor Cores require dimensions to be multiples of 16
    if out_features % 16 != 0 or in_features % 16 != 0:
        print(f"⚠️ Skipping {layer_name}: Shape [{out_features}, {in_features}] fails (16, 16) hardware alignment.")
        # Cast to half precision anyway so it matches the rest of the network's dtype
        linear_layer.weight.data = linear_layer.weight.data.half()
        if linear_layer.bias is not None:
            linear_layer.bias.data = linear_layer.bias.data.half()
        return

    # 2. A100 Sparse Tensor Cores require FP16.
    linear_layer.weight.data = linear_layer.weight.data.half()
    if linear_layer.bias is not None:
        linear_layer.bias.data = linear_layer.bias.data.half()

    # 3. Reshape into blocks of 4
    W = linear_layer.weight.data.clone()
    shape = W.shape
    W_4 = W.abs().view(-1, 4)

    # 4. Mask the 2 smallest magnitudes in each block
    _, indices = torch.topk(W_4, k=2, dim=-1, largest=False)
    W_4.scatter_(1, indices, 0)
    linear_layer.weight.data = W_4.view(shape)

    # 5. Compress to PyTorch Semi-Structured Sparse Tensor
    linear_layer.weight = torch.nn.Parameter(to_sparse_semi_structured(linear_layer.weight.data))
    print(f"✅ {layer_name}: 2:4 Structured Sparsity applied. [{out_features}, {in_features}] Tensor Cores engaged.")


# Instantiate and push to A100
# Note: We do NOT call .half() on the whole model here, the function handles it cleanly
snn_pruned = RecurrentSNN().to(DEVICE)
snn_pruned.load_state_dict(snn_model.state_dict())

# Apply the hardware-compliant pruning
print("--- Initializing Hardware-Aware Sparsity ---")
apply_2_4_sparsity(snn_pruned.fc1, layer_name="fc1")
apply_2_4_sparsity(snn_pruned.fc2, layer_name="fc2")
print("--------------------------------------------")

--- Initializing Hardware-Aware Sparsity ---
✅ fc1: 2:4 Structured Sparsity applied. [128, 64] Tensor Cores engaged.
⚠️ Skipping fc2: Shape [4, 128] fails (16, 16) hardware alignment.
--------------------------------------------


/usr/local/lib/python3.12/dist-packages/torch/sparse/semi_structured.py:571: UserWarning: The PyTorch API of SparseSemiStructuredTensor is in prototype stage and will change in the near future. Please open a Github issue for features requests and see our documentation on the torch.sparse module for further information about the project.
  return cls(


## ⚡ 5. Inference Latency Benchmark
Measures wall-clock time per batch (averaged over multiple runs with warmup).

In [ ]:
def inference_benchmark(model, name, num_warmup=10, num_runs=50):
    model.eval()
    model.to(DEVICE)

    # Cast to Half-Precision for A100 Tensor Cores
    if "Pruned" in name:
        model = model.half()

    # Fuse the CUDA kernels to eliminate Python dispatch overhead
    print(f"Compiling {name} execution graph...")
    compiled_model = torch.compile(model, mode="reduce-overhead")

    sample_batch = next(iter(test_loader))
    inputs = sample_batch['input_ids'].to(DEVICE)
    bs = inputs.size(0)

    # Warmup (Compilation happens during the first forward pass)
    with torch.no_grad():
        for _ in range(num_warmup):
            compiled_model(inputs)

    # Timed runs
    times = []
    with torch.no_grad():
        for _ in range(num_runs):
            torch.cuda.synchronize()
            t0 = time.time()
            compiled_model(inputs)
            torch.cuda.synchronize()
            times.append(time.time() - t0)

    # ... (Keep the rest of your latency calculation code identical) ...

## 🎯 6. Test Set Evaluation
Overall accuracy + per-class breakdown + confusion matrix.

In [ ]:
import torch

def evaluate_test(model, name):
    model.eval()
    model.to(DEVICE)

    # Initialize a GPU tensor for the confusion matrix
    conf_matrix = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.int64, device=DEVICE)

    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.float16):
        for batch in test_loader:
            inputs = batch['input_ids'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            logits, _ = model(inputs)
            preds = logits.argmax(1)

            # Vectorized Confusion Matrix update (Runs entirely on GPU in C++/CUDA)
            # labels * NUM_CLASSES + preds creates a unique 1D index for every [true, pred] pair
            conf_matrix += torch.bincount(
                labels * NUM_CLASSES + preds,
                minlength=NUM_CLASSES**2
            ).view(NUM_CLASSES, NUM_CLASSES)

    # Calculate metrics directly from the confusion matrix
    total = conf_matrix.sum().item()
    correct = conf_matrix.diag().sum().item()
    overall_acc = correct / total

    # Per-class accuracy: diagonal / row_sums
    class_totals = conf_matrix.sum(dim=1)
    class_corrects = conf_matrix.diag()

    per_class = {}
    for i in range(NUM_CLASSES):
        # Prevent division by zero
        denom = max(class_totals[i].item(), 1)
        per_class[CLASS_NAMES[i]] = round(class_corrects[i].item() / denom, 4)

    print(f"  [{name:12s}]  Accuracy: {overall_acc:.4f}")
    for cls, acc in per_class.items():
        print(f"      {cls:10s}: {acc:.4f}")

    return {
        'name': name,
        'test_accuracy': round(overall_acc, 4),
        'per_class_accuracy': per_class,
        'confusion_matrix': conf_matrix.cpu().tolist()
    }

## 📋 7. Complete Metrics Summary
Copy the JSON output below — it will be used to generate charts.

In [ ]:
def count_nonzero_weights(model):
    total_weights = 0
    alive_weights = 0
    for name, param in model.named_parameters():
        try:
            # Attempt standard dense evaluation
            total_weights += param.numel()
            alive_weights += (param != 0).sum().item()
        except NotImplementedError:
            # Catch the A100 cuSPARSELt tensor error
            # We strictly enforced 2:4 sparsity, meaning exactly 50% are alive
            m, n = param.shape
            elements = m * n
            total_weights += elements
            alive_weights += elements // 2
    return total_weights, alive_weights

In [ ]:
!pip install -q snntorch datasets transformers matplotlib seaborn numpy

In [ ]:
import time
import json
import numpy as np
import torch
import torch.nn as nn
from snntorch import surrogate
import snntorch as snn

print("🔧 Refactoring SNN Architecture for CUDA Graph Compilation...")

# 1. Save trained weights
trained_state = snn_model.state_dict()

# 2. Functional RecurrentSNN (Compiler Safe)
class RecurrentSNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        spike_grad = surrogate.fast_sigmoid(slope=25)
        self.fc1 = nn.Linear(EMBED_DIM, HIDDEN_DIM)
        self.lif1 = snn.Leaky(beta=0.9, spike_grad=spike_grad, init_hidden=False)
        self.fc2 = nn.Linear(HIDDEN_DIM, NUM_CLASSES)
        self.lif2 = snn.Leaky(beta=0.9, spike_grad=spike_grad, init_hidden=False)

    def forward(self, x):
        B, S = x.shape
        emb = self.emb(x)

        mem1 = torch.zeros(B, HIDDEN_DIM, device=x.device, dtype=emb.dtype)
        mem2 = torch.zeros(B, NUM_CLASSES, device=x.device, dtype=emb.dtype)
        spk2_sum = torch.zeros(B, NUM_CLASSES, device=x.device, dtype=emb.dtype)

        # 🟢 Minor fix: ensure our counter inherits the exact precision of the graph
        total_ops = torch.tensor(0.0, device=x.device, dtype=emb.dtype)

        for t in range(S):
            cur = self.fc1(emb[:, t, :])
            spk1, mem1 = self.lif1(cur, mem1)
            cur_out = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur_out, mem2)

            spk2_sum = spk2_sum + spk2

            total_ops = total_ops + spk1.sum() * HIDDEN_DIM
            total_ops = total_ops + spk2.sum() * NUM_CLASSES

        return spk2_sum, total_ops

# 3. Load Models
snn_model = RecurrentSNN().to(DEVICE)
snn_model.load_state_dict(trained_state)

snn_pruned = RecurrentSNN().to(DEVICE)
snn_pruned.load_state_dict(trained_state)

# 4. Re-apply Hardware Sparsity
print("--- Re-initializing Hardware-Aware Sparsity ---")
apply_2_4_sparsity(snn_pruned.fc1, layer_name="fc1")
apply_2_4_sparsity(snn_pruned.fc2, layer_name="fc2")

# 5. The Autocast-Protected Benchmark Loop
def inference_benchmark(model, name, num_warmup=10, num_runs=50):
    model.eval()
    model.to(DEVICE)

    # 🟢 FIX: Removed the manual model.half() cast.
    print(f"Compiling {name} execution graph...")
    compiled_model = torch.compile(model, mode="reduce-overhead")

    sample_batch = next(iter(test_loader))
    inputs = sample_batch['input_ids'].to(DEVICE)
    bs = inputs.size(0)

    # 🟢 FIX: Wrap the ENTIRE compilation and execution in autocast
    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.float16):
        # Warmup
        for _ in range(num_warmup):
            compiled_model(inputs)

        # Timed runs
        times = []
        for _ in range(num_runs):
            torch.cuda.synchronize()
            t0 = time.time()
            compiled_model(inputs)
            torch.cuda.synchronize()
            times.append(time.time() - t0)

    avg_time = np.mean(times)
    avg_batch_ms = avg_time * 1000
    per_sample_ms = avg_batch_ms / bs
    throughput = bs / avg_time

    print(f"  [{name:12s}]  {avg_batch_ms:.2f} ms/batch  │  {throughput:,.0f} samp/s")
    return {
        'name': name,
        'avg_batch_ms': round(avg_batch_ms, 2),
        'per_sample_ms': round(per_sample_ms, 3),
        'throughput_sps': round(throughput, 2)
    }

print("\n🚀 RUNNING COMPILED INFERENCE BENCHMARKS...")
print("─" * 50)
inf_tf     = inference_benchmark(transformer_model, "Transformer")
inf_snn    = inference_benchmark(snn_model,         "SNN")
inf_pruned = inference_benchmark(snn_pruned,        "Pruned SNN")

print("\n🎯 RUNNING TEST SET EVALUATIONS...")
print("─" * 50)
test_tf     = evaluate_test(transformer_model, "Transformer")
test_snn    = evaluate_test(snn_model,         "SNN")
test_pruned = evaluate_test(snn_pruned,        "Pruned SNN")

# 6. JSON Export
total_w_snn, alive_w_snn = count_nonzero_weights(snn_model)
total_w_pr,  alive_w_pr  = count_nonzero_weights(snn_pruned)

metrics = {
    "config": {
        "device": str(DEVICE), "batch_size": BATCH_SIZE, "seq_len": MAX_SEQ_LEN
    },
    "training": {
        "transformer": history_tf, "snn": history_snn, "pruned_snn": history_snn
    },
    "inference": {
        "transformer": inf_tf, "snn": inf_snn, "pruned_snn": inf_pruned,
    },
    "test_evaluation": {
        "transformer": test_tf, "snn": test_snn, "pruned_snn": test_pruned,
    },
    "model_size": {
        "transformer_params": count_parameters(transformer_model),
        "snn_params": count_parameters(snn_model),
        "snn_total_weights": total_w_snn,
        "snn_alive_weights": alive_w_snn,
        "pruned_total_weights": total_w_pr,
        "pruned_alive_weights": alive_w_pr,
    }
}

print("\n\n" + "=" * 60)
print("  📋 JSON METRICS — Copy everything between the markers")
print("=" * 60)
print("--- METRICS_JSON_START ---")
print(json.dumps(metrics, indent=2))
print("--- METRICS_JSON_END ---")

🔧 Refactoring SNN Architecture for CUDA Graph Compilation...
--- Re-initializing Hardware-Aware Sparsity ---
✅ fc1: 2:4 Structured Sparsity applied. [128, 64] Tensor Cores engaged.
⚠️ Skipping fc2: Shape [4, 128] fails (16, 16) hardware alignment.

🚀 RUNNING COMPILED INFERENCE BENCHMARKS...
──────────────────────────────────────────────────
Compiling Transformer execution graph...
  [Transformer ]  0.25 ms/batch  │  251,547 samp/s
Compiling SNN execution graph...
  [SNN         ]  45.71 ms/batch  │  1,400 samp/s
Compiling Pruned SNN execution graph...
  [Pruned SNN  ]  79.68 ms/batch  │  803 samp/s

🎯 RUNNING TEST SET EVALUATIONS...
──────────────────────────────────────────────────
  [Transformer ]  Accuracy: 0.7580
      World     : 0.7534
      Sports    : 0.7738
      Business  : 0.7372
      Sci/Tech  : 0.7646
  [SNN         ]  Accuracy: 0.6760
      World     : 0.7808
      Sports    : 0.6882
      Business  : 0.6058
      Sci/Tech  : 0.6206
  [Pruned SNN  ]  Accuracy: 0.2555
   